### Reglerentwurf nach Reinisch

Bei Reinisch ist das Ziel, durch einen Regler eine **$\mathrm{IT}_1$-offene Kette** zu realisieren.
Große Zeitkonstanten werden gekürzt, um ein maximal schnell reagierendes System zu erhalten.
Stelle die Zeitkonstante $T_N$ so ein, dass du das gewünschte Verhalten bekommst.

In [3]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import control as ct
import ipywidgets as widgets
from IPython.display import display, Markdown

# Feste Streckenparameter
KS_FIXED = 2.0
T1_FIXED = 2.0  # Große Zeitkonstante (soll kompensiert werden)
T2_FIXED = 0.5  # Kleine ungekürzte Zeitkonstante

def display_reinisch_formulas(show_controller, a, TN):
    """Generiert dynamische LaTeX-Formeln fehlerfrei über Markdown."""
    latex_GS = f"G_S(s) = \\frac{{{KS_FIXED}}}{{({T1_FIXED:.1f}s + 1)({T2_FIXED:.1f}s + 1)}}"

    if not show_controller:
        display(Markdown(f"$$\\mathbf{{Strecke}}\\ G_S(s): \\quad {latex_GS}$$"))
        return

    # KR-Berechnung nach Reinisch
    KR = TN / (a * T2_FIXED * KS_FIXED)

    is_perfect_cancel = abs(TN - T1_FIXED) < 1e-3

    # Regler symbolisch und mit Zahlen
    latex_GR = f"G_R(s) = K_R \\cdot \\frac{{T_N s + 1}}{{T_N s}} = {KR:.2f} \\cdot \\frac{{{TN:.2f}s + 1}}{{{TN:.2f}s}}"

    # Dynamischer Aufbau der gekürzten Formel für G_0(s) (a bleibt explizit sichtbar)
    if is_perfect_cancel:
        latex_G0 = (
            f"G_0(s) = {KR:.2f} \\cdot \\frac{{{TN:.2f}s + 1}}{{{TN:.2f}s}} \\cdot "
            f"\\frac{{{KS_FIXED}}}{{({T1_FIXED:.1f}s + 1)({T2_FIXED:.1f}s + 1)}} "
            f"= \\mathbf{{\\frac{{1}}{{{a:.1f} \\cdot {T2_FIXED:.1f} s ({T2_FIXED:.1f}s + 1)}}}}"
        )
    else:
        latex_G0 = (
            f"G_0(s) = {KR:.2f} \\cdot \\frac{{{TN:.2f}s + 1}}{{{TN:.2f}s}} \\cdot "
            f"\\frac{{{KS_FIXED}}}{{({T1_FIXED:.1f}s + 1)({T2_FIXED:.1f}s + 1)}} "
            f"= \\mathbf{{\\frac{{{TN:.2f}s + 1}}{{{a:.1f} \\cdot {T2_FIXED:.1f} s ({T1_FIXED:.1f}s + 1)({T2_FIXED:.1f}s + 1)}}}}"
        )

    latex_output = f"""
$$
\\begin{{aligned}}
\\mathbf{{Strecke}}\\ G_S(s): \\quad & {latex_GS} \\\\[8pt]
\\mathbf{{Regler}}\\ G_R(s): \\quad & {latex_GR} \\\\[8pt]
\\mathbf{{Offene\\ Kette:}} \\quad & {latex_G0}
\\end{{aligned}}
$$
"""
    display(Markdown(latex_output))

def plot_step_response_reinisch(show_controller=False, a=2.0, TN=2.0):
    display_reinisch_formulas(show_controller, a, TN)

    t = np.linspace(0, 15, 1000)

    # Streckenübertragungsfunktion GS(s) [PT2]
    num_S = [KS_FIXED]
    den_S = [T1_FIXED * T2_FIXED, T1_FIXED + T2_FIXED, 1]
    GS = ct.tf(num_S, den_S)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.axhline(1.0, color='black', ls='--', lw=1.2, label=r'_nolegend_')

    if not show_controller:
        t_out, y_S = ct.step_response(GS, t)
        ax.plot(t_out, y_S, 'b-', lw=2.5, label=r'Ungeregelte Strecke $y_S(t)$')
    else:
        KR = TN / (a * T2_FIXED * KS_FIXED)

        # Realer geschlossener Regelkreis
        GR = ct.tf([KR * TN, KR], [TN, 0])
        G0 = GR * GS
        Gw = ct.feedback(G0, 1)
        t_out, y_closed = ct.step_response(Gw, t)

        # Ideales Soll-Verhalten für G0(s) = 1 / (a * T2 * s * (1 + s * T2))
        G0_ideal = ct.tf([1], [a * (T2_FIXED**2), a * T2_FIXED, 0])
        Gw_ideal = ct.feedback(G0_ideal, 1)
        _, y_ideal = ct.step_response(Gw_ideal, t)

        # Plots
        ax.plot(t_out, y_closed, 'r-', lw=2.5, label=rf'Ausgang')
        ax.plot(t_out, y_ideal, 'g--', lw=1.8, label=rf'Soll-$IT_1$-Verhalten')

    ax.set_title(r"Sprungantwort im Zeitbereich", fontsize=12)
    ax.set_xlabel(r" $t$ [s]", fontsize=10)
    ax.set_ylabel(r" $y(t)$", fontsize=10)
    ax.grid(True, ls="-", alpha=0.5)
    ax.legend(fontsize=10, loc="lower right")
    ax.set_ylim(-0.1, 2.0)
    ax.set_xlim(0, 15)

    plt.show()

# Interaktive Steuerung ausschließlich mit a und TN
w_show = widgets.Checkbox(value=False, description='Regler zuschalten')
w_a    = widgets.FloatSlider(value=2.0, min=0.5, max=3.0, step=0.2, description='Faktor a:')
w_TN   = widgets.FloatSlider(value=1.4, min=0.1, max=5.0, step=0.1, description='T_N [s]:')

interactive_plot = widgets.interactive_output(
    plot_step_response_reinisch,
    {'show_controller': w_show, 'a': w_a, 'TN': w_TN}
)

controls = widgets.VBox([
    w_show,
    widgets.HBox([w_a, w_TN])
])

display(interactive_plot, controls)

Output()

### Summenzeitkonstante
Mithilfe der Summenzeitkonstante ist es möglich die Ordnung höherer Systeme zu reduzieren. Dabei werden die kleineren Zeitkonstanten addiert.
 Die Flächen über der Sprungantwort für PT3, sowohl als auch für die Summenzeitkonstante sind gleich. Über die Fläche lässt sich, bei einem unbekannten System, ebenfalls die Summenzeitkonstante berechnen.



In [4]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
from scipy.integrate import trapezoid
import ipywidgets as widgets
from IPython.display import display

# Feste Parameter
K_FIXED = 1.0
T_MAX_FIXED = 50.0  # Fester Zeitbereich für die X-Achse (0 bis 50 Sekunden)

def plot_summenzeitkonstante(T1=2.0, T2=1.0, T3=0.5):
    # Fester Zeitvektor
    T_sum = T1 + T2 + T3
    t = np.linspace(0, T_MAX_FIXED, 1000)

    # 1. Originales PT3-System: G(s) = 1 / ((T1*s + 1)(T2*s + 1)(T3*s + 1))
    den1 = [T1, 1]
    den2 = [T2, 1]
    den3 = [T3, 1]
    den_orig = np.poly1d(den1) * np.poly1d(den2) * np.poly1d(den3)

    sys_orig = signal.TransferFunction([K_FIXED], den_orig.coefficients)
    _, y_orig = signal.step(sys_orig, T=t)

    # 2. Ersatz-PT1-System mit T_sigma
    sys_approx = signal.TransferFunction([K_FIXED], [T_sum, 1])
    _, y_approx = signal.step(sys_approx, T=t)

    # Diagramm erstellen
    fig, ax = plt.subplots(figsize=(9, 4.5))

    # Stationärer Endwert K = 1
    ax.axhline(K_FIXED, color='black', linestyle='--', linewidth=1.2, label=r'_nolegend_')

    # Kurven zeichnen
    ax.plot(t, y_orig, 'b-', linewidth=2.5, label=f'$PT_3$ ')
    ax.plot(t, y_approx, 'r--', linewidth=2.0, label=f'$PT_1$ mit Summenzeitkonstante')

    # Flächen dauerhaft einfärben
    ax.fill_between(t, y_orig, K_FIXED, color='blue', alpha=0.15, label=f'_nolegend_')
    ax.fill_between(t, y_approx, K_FIXED, color='red', alpha=0.15, label=f'_nolegend_')

    # Plot-Formatierung
    ax.set_title(r"Summenzeitkonstante $T_\Sigma$ und Flächengleichheit", fontsize=12)
    ax.set_xlabel(r"Zeit $t$ [s]", fontsize=10)
    ax.set_ylabel(r"Ausgangsgröße $y(t)$", fontsize=10)
    ax.grid(True, linestyle='-', alpha=0.5)
    ax.legend(fontsize=9, loc="lower right")

    # FESTE ACHSENGRENZEN (verhindert das Springen des Diagramms)
    ax.set_xlim(0, T_MAX_FIXED)
    ax.set_ylim(-0.05, 1.25)

    plt.show()

# Interaktive Schieberegler ausschließlich für T1, T2 und T3
w_T1 = widgets.FloatSlider(value=2.0, min=0.1, max=5.0, step=0.1, description='T1 [s]:')
w_T2 = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='T2 [s]:')
w_T3 = widgets.FloatSlider(value=0.5, min=0.0, max=5.0, step=0.1, description='T3 [s]:')

interactive_widget = widgets.interactive_output(
    plot_summenzeitkonstante,
    {'T1': w_T1, 'T2': w_T2, 'T3': w_T3}
)

controls = widgets.HBox([w_T1, w_T2, w_T3])

display(interactive_widget, controls)

Output()

### Betragsoptimum
Das Ziel des Betragsoptimums ist es, den Betrag der Führungsübertragungsfunktion so lang und nah wie möglich 1 (0 dB) werden zu lassen.
Warum ist es nicht möglich für höhere Frequenzen dieses Verhalten zu erreichen?



In [5]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
import ipywidgets as widgets
from IPython.display import display, Markdown

# Feste Streckenparameter
KS_FIXED = 1.0  # Streckenverstärkung
T1_FIXED = 2.0  # Große Zeitkonstante (wird durch TN = T1 kompensiert)

# Feste Achsengrenzen
W_MIN, W_MAX = 0.1, 50.0   # Kreisfrequenzbereich [rad/s]
T_MAX_FIXED = 10.0         # Zeitbereich [s]

def display_formulas(a, T2, KR):
    """Generiert dynamisch die Übertragungsfunktionen mit aktuellen Zahlenwerten."""
    aT2 = a * T2
    aT2_sq = a * (T2**2)

    latex_text = f"""
$$
\\begin{{aligned}}
\\mathbf{{Strecke}}\\ G_S(s): \\quad & \\frac{{{KS_FIXED:.1f}}}{{({T1_FIXED:.1f}s + 1)({T2:.2f}s + 1)}} \\\\[6pt]
\\mathbf{{PI-Regler}}\\ G_R(s): \\quad & {KR:.2f} \\cdot \\frac{{{T1_FIXED:.1f}s + 1}}{{{T1_FIXED:.1f}s}} \\\\[6pt]
\\mathbf{{Offene\\ Kette}}\\ G_0(s): \\quad & \\frac{{1}}{{{a:.1f} \\cdot {T2:.2f}s ({T2:.2f}s + 1)}} = \\frac{{1}}{{{aT2_sq:.3f} s^2 + {aT2:.2f} s}} \\\\[6pt]
\\mathbf{{Geschlossener\\ Regelkreis}}\\ G_w(s): \\quad & \\mathbf{{\\frac{{1}}{{{aT2_sq:.3f} s^2 + {aT2:.2f} s + 1}}}}
\\end{{aligned}}
$$
"""
    display(Markdown(latex_text))

def plot_pi_pt2_betragsoptimum(a=2.0, T2=0.5):
    # 1. Parameter nach Betragsoptimum
    TN = T1_FIXED                       # Pol-Nullstellen-Kompensation
    KR = TN / (a * KS_FIXED * T2)       # Reglerverstärkung

    # Dynamic Markdown Output
    display_formulas(a, T2, KR)

    # 2. Übertragungsfunktionen definieren
    num_S = [KS_FIXED]
    den_S = [T1_FIXED * T2, T1_FIXED + T2, 1.0]

    num_R = [KR * TN, KR]
    den_R = [TN, 0.0]

    num_0 = np.polymul(num_R, num_S)
    den_0 = np.polymul(den_R, den_S)

    den_w = np.polyadd(den_0, num_0)
    sys_w = signal.TransferFunction(num_0, den_w)

    # 3. Frequenzgang berechnen (in dB)
    omega = np.logspace(np.log10(W_MIN), np.log10(W_MAX), 1000)
    _, mag_complex = signal.freqs(num_0, den_w, worN=omega)
    mag_db = 20 * np.log10(np.abs(mag_complex))

    # 4. Sprungantwort berechnen
    t = np.linspace(0, T_MAX_FIXED, 1000)
    _, y_t = signal.step(sys_w, T=t)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.0))

    # Farbmarkierung für a = 2.0 (Betragsoptimum)
    line_color = 'green' if abs(a - 2.0) < 1e-2 else ('red' if a < 2.0 else 'blue')

    # --- Subplot 1: Betragsverlauf im Frequenzbereich (in dB) ---
    ax1.axhline(0.0, color='black', linestyle='--', linewidth=1.2, label=r'_nolegend_')
    ax1.semilogx(omega, mag_db, color=line_color, linewidth=2.5,
                 label=f'Übertragunsfunktion')

    ax1.set_title(r"Betragsverlauf im Frequenzbereich", fontsize=12)
    ax1.set_xlabel(r"Kreisfrequenz $\omega$ [rad/s]", fontsize=10)
    ax1.set_ylabel(r"Betrag $|G_w(j\omega)|$ [dB]", fontsize=10)
    ax1.grid(True, which='both', linestyle='-', alpha=0.5)
    ax1.legend(fontsize=9, loc="lower left")

    # FESTE ACHSEN (Frequenzbereich)
    ax1.set_xlim(W_MIN, W_MAX)
    ax1.set_ylim(-20.0, 5.0)

    # --- Subplot 2: Sprungantwort im Zeitbereich ---
    ax2.axhline(1.0, color='black', linestyle='--', linewidth=1.2, label=r'_nolegend_')
    ax2.plot(t, y_t, color=line_color, linewidth=2.5, label=f'Sprungantwort')

    ax2.set_title(r"Sprungantwort im Zeitbereich", fontsize=12)
    ax2.set_xlabel(r"Zeit $t$ [s]", fontsize=10)
    ax2.set_ylabel(r"Regelgröße $y(t)$", fontsize=10)
    ax2.grid(True, linestyle='-', alpha=0.5)
    ax2.legend(fontsize=9, loc="lower right")

    # FESTE ACHSEN (Zeitbereich)
    ax2.set_xlim(0, T_MAX_FIXED)
    ax2.set_ylim(-0.05, 1.6)

    plt.tight_layout()
    plt.show()

# Interaktive Schieberegler für a und T2 (=T)
w_a  = widgets.FloatSlider(value=2.0, min=0.5, max=5.0, step=0.1, description='Faktor a:')
w_T2 = widgets.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05, description='T2 (=T) [s]:')

interactive_widget = widgets.interactive_output(
    plot_pi_pt2_betragsoptimum,
    {'a': w_a, 'T2': w_T2}
)

controls = widgets.HBox([w_a, w_T2])

display(interactive_widget, controls)

Output()

### Polvorgabe
Die Lage des dominanten Polpaares wird begrenzt durch
* Überschwingweite
* Überschwingzeit
* Beruhigungszeit


Mit diesen Forderungen ergibt sich ein Zielgebiet für die Polvergabe.
Sind die Pole ausgewählt, lässt sich der Regler berechnen.

In [6]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
import ipywidgets as widgets
from IPython.display import display, Markdown

# --- Feste Streckenparameter (PT1-Glied laut Beispiel 3.7) ---
KS_FIXED = 1.0  # Streckenverstärkung K_S
T1_FIXED = 2.0  # Zeitkonstante T_1

# --- Fest vorgegebenes Polgebiet ---
DELTA_MIN, DELTA_MAX = 0.6, 2.5       # Abklingkoeffizient delta_e (Realteil -delta)
OMEGA_MIN, OMEGA_MAX = 1.0, 4.0       # Eigenfrequenz omega_e (Imaginärteil)
PHI_MIN_DEG, PHI_MAX_DEG = 30.0, 60.0 # Dämpfungswinkel phi

def is_pole_in_region(delta, omega):
    """Prüft, ob der Pol (-delta + j*omega) im zulässigen Bereich liegt."""
    phi_deg = np.degrees(np.arctan2(omega, delta))
    in_delta = DELTA_MIN <= delta <= DELTA_MAX
    in_omega = OMEGA_MIN <= omega <= OMEGA_MAX
    in_phi = PHI_MIN_DEG <= phi_deg <= PHI_MAX_DEG
    return in_delta and in_omega and in_phi

def display_pole_formulas(delta, omega):
    """Berechnung & Anzeige nach den Formeln aus Beispiel 3.7."""
    omega0_sq = delta**2 + omega**2
    omega0 = np.sqrt(omega0_sq)
    T = 1.0 / omega0
    D = delta / omega0

    TN = (2 * D * T * T1_FIXED - T**2) / T1_FIXED
    KP = (2 * D * T1_FIXED - T) / (KS_FIXED * T)

    latex_text = f"""
$$
\\begin{{aligned}}
\\mathbf{{Gewähltes\\ Polpaar}}\\ s_{{1,2}}: \\quad & -{delta:.2f} \\pm j{omega:.2f} \\\\[6pt]
\\mathbf{{Dämpfung}}\\ D, \\mathbf{{Zeitkonst.}}\\ T: \\quad & D = {D:.3f}, \\quad T = {T:.3f}\\,\\mathrm{{s}} \\\\[6pt]
\\mathbf{{Berechneter\\ PI\\text{{-}}Regler}}\\ G_R(s): \\quad & K_P = {KP:.3f}, \\quad T_N = {TN:.3f}\\,\\mathrm{{s}} \\\\[6pt]
& G_R(s) = {KP:.3f} \\cdot \\left( 1 + \\frac{{1}}{{{TN:.3f} s}} \\right)
\\end{{aligned}}
$$
"""
    display(Markdown(latex_text))

def plot_polvorgabe(delta=1.2, omega=2.0):
    in_region = is_pole_in_region(delta, omega)
    display_pole_formulas(delta, omega)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.0))
    line_color = 'green' if in_region else 'red'

    # =========================================================================
    # --- Subplot 1: s-Ebene mit Polgebiet und Polpaar ---
    # =========================================================================
    d_grid = np.linspace(DELTA_MIN, DELTA_MAX, 200)
    w_grid = np.linspace(OMEGA_MIN, OMEGA_MAX, 200)
    D_mesh, W_mesh = np.meshgrid(d_grid, w_grid)
    PHI = np.degrees(np.arctan2(W_mesh, D_mesh))

    mask = (PHI >= PHI_MIN_DEG) & (PHI <= PHI_MAX_DEG)

    ax1.contourf(-D_mesh, W_mesh, np.where(mask, 1, 0), levels=[0.5, 1.5], colors=['#b0c4de'], alpha=0.6)
    ax1.contourf(-D_mesh, -W_mesh, np.where(mask, 1, 0), levels=[0.5, 1.5], colors=['#b0c4de'], alpha=0.6)

    # Führungslinien / Grenzen
    ax1.axvline(-DELTA_MIN, color='gray', linestyle=':', linewidth=1)
    ax1.axvline(-DELTA_MAX, color='gray', linestyle=':', linewidth=1)
    ax1.axhline(OMEGA_MIN, color='gray', linestyle=':', linewidth=1)
    ax1.axhline(OMEGA_MAX, color='gray', linestyle=':', linewidth=1)
    ax1.axhline(-OMEGA_MIN, color='gray', linestyle=':', linewidth=1)
    ax1.axhline(-OMEGA_MAX, color='gray', linestyle=':', linewidth=1)

    # Strahlen für phi_min und phi_max
    d_ray = np.array([0, 4.0])
    ax1.plot(-d_ray, d_ray * np.tan(np.radians(PHI_MIN_DEG)), 'gray', linestyle=':', linewidth=1)
    ax1.plot(-d_ray, d_ray * np.tan(np.radians(PHI_MAX_DEG)), 'gray', linestyle=':', linewidth=1)
    ax1.plot(-d_ray, -d_ray * np.tan(np.radians(PHI_MIN_DEG)), 'gray', linestyle=':', linewidth=1)
    ax1.plot(-d_ray, -d_ray * np.tan(np.radians(PHI_MAX_DEG)), 'gray', linestyle=':', linewidth=1)

    # Beschriftung der Winkel
    r_text_min, r_text_max = 3.2, 2.8
    x_phi_min = -r_text_min * np.cos(np.radians(PHI_MIN_DEG))
    y_phi_min = r_text_min * np.sin(np.radians(PHI_MIN_DEG))
    x_phi_max = -r_text_max * np.cos(np.radians(PHI_MAX_DEG))
    y_phi_max = r_text_max * np.sin(np.radians(PHI_MAX_DEG))

    ax1.text(x_phi_min, y_phi_min, r'$\varphi_{\mathrm{min}}$', fontsize=10, color='white',
             ha='right', va='bottom', rotation=-PHI_MIN_DEG)
    ax1.text(x_phi_max, y_phi_max, r'$\varphi_{\mathrm{max}}$', fontsize=10, color='white',
             ha='right', va='bottom', rotation=-PHI_MAX_DEG)

    # Achsenkreuze
    ax1.axhline(0, color='black', linewidth=1.2)
    ax1.axvline(0, color='black', linewidth=1.2)

    # Pole plotten
    ax1.plot([-delta, -delta], [omega, -omega], 'x', color=line_color,
             markersize=10, markeredgewidth=3, label='Pole $s_{1,2}$')

    # Beschriftungen
    ax1.text(-DELTA_MIN, -0.3, r'$-\delta_{e,\mathrm{min}}$', ha='center', fontsize=9)
    ax1.text(-DELTA_MAX, -0.3, r'$-\delta_{e,\mathrm{max}}$', ha='center', fontsize=9)
    ax1.text(0.1, OMEGA_MIN, r'$\omega_{e,\mathrm{min}}$', va='center', fontsize=9)
    ax1.text(0.1, OMEGA_MAX, r'$\omega_{e,\mathrm{max}}$', va='center', fontsize=9)

    ax1.set_title(r"Polgebiet in der $s$-Ebene", fontsize=12)
    ax1.set_xlabel(r"Realteil $\mathrm{Re}\{s\}$", fontsize=10)
    ax1.set_ylabel(r"Imaginärteil $\mathrm{Im}\{s\}$", fontsize=10)
    ax1.set_xlim(-3.5, 0.5)
    ax1.set_ylim(-5.0, 5.0)
    ax1.grid(True, linestyle='-', alpha=0.3)
    ax1.legend(loc="upper left", fontsize=9)

    # =========================================================================
    # --- Subplot 2: Sprungantwort des geschlossenen Regelkreises ---
    # =========================================================================
    omega0_sq = delta**2 + omega**2
    omega0 = np.sqrt(omega0_sq)
    T = 1.0 / omega0
    D = delta / omega0

    TN = (2 * D * T * T1_FIXED - T**2) / T1_FIXED
    KP = (2 * D * T1_FIXED - T) / (KS_FIXED * T)

    num_open = [KP * KS_FIXED * TN, KP * KS_FIXED]
    den_open = [TN * T1_FIXED, TN, 0]

    num_closed = num_open
    den_closed = [TN * T1_FIXED, TN + KP * KS_FIXED * TN, KP * KS_FIXED]

    sys_w = signal.TransferFunction(num_closed, den_closed)

    t = np.linspace(0, 10.0, 1000)
    _, y_t = signal.step(sys_w, T=t)

    ax2.axhline(1.0, color='black', linestyle='--', linewidth=1.2)
    ax2.plot(t, y_t, color=line_color, linewidth=2.5, label='Sprungantwort $y(t)$')

    ax2.set_title(r"Sprungantwort $y(t)$", fontsize=12)
    ax2.set_xlabel(r"Zeit $t$ [s]", fontsize=10)
    ax2.set_ylabel(r"Regelgröße $y(t)$", fontsize=10)
    ax2.set_xlim(0, 10.0)
    ax2.set_ylim(-0.05, 1.6)
    ax2.grid(True, linestyle='-', alpha=0.5)
    ax2.legend(loc="lower right", fontsize=9)

    plt.tight_layout()
    plt.show()
    plt.close(fig)  # Verhindert die doppelte Ausgabe

# --- Interaktive Slider ---
w_delta = widgets.FloatSlider(value=1.2, min=0.1, max=3.0, step=0.1, description='delta:')
w_omega = widgets.FloatSlider(value=2.0, min=0.1, max=4.5, step=0.1, description='omega_e:')

interactive_widget = widgets.interactive_output(
    plot_polvorgabe,
    {'delta': w_delta, 'omega': w_omega}
)

controls = widgets.HBox([w_delta, w_omega])
display(interactive_widget, controls)

Output()